# PhishLang-Web: leakage-free MobileBERT training (Colab free tier)

CS 8027 research project · Masroor Ul Hasan Syed, Deepthi Kondreddy

What this notebook does, in order:

1. Downloads PhishLang's parsed pages and its released model from the upstream repo at commit `6b72838` (about 440 MB) and checks every file against its git hash.
2. Removes empty pages, pages whose exact text appears in both classes, and exact duplicates.
3. **E0:** measures each page's length in tokens: how much of a page does a 128-token training window see?
4. Builds a leakage-free split (63% train / 7% validation / 30% test), checks that no page text crosses splits, and counts test pages whose 128-token model input is identical to some training page's input.
5. **E1 (sanity check only):** scores the *released* model on our test split. This is **not** a held-out score, because the released model was probably trained on most of these pages. It only shows that labels and tokenization match upstream.
6. **E2:** retrains MobileBERT with upstream's hyper-parameters (lr 2e-5, batch 32 × 4 gradient accumulation, 10 epochs, first 128 tokens, weight decay 0.01, FP32, Trainer defaults otherwise) and scores it on the test split.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**, and allow Google Drive access when asked. The training progress bar shows the time left.
If Colab disconnects, reconnect and **Run all** again. The data is downloaded again, finished steps are skipped, and training resumes from the last finished epoch.

**Local run instead (WSL Ubuntu):** `bash notebooks/run_local_wsl.sh` from inside WSL runs this same notebook with the data in `~/phish-research` (Linux disk), no network access and the Windows drives hidden.

**Safety rules (phishing data):**
- The pages are real phishing HTML/JavaScript converted to text. They live only on this VM's disk (`/content/phish-data`), which is wiped when the runtime is deleted.
- Never copy `/content/phish-data` to Google Drive or download it to your PC. Drive (`MyDrive/phishlang_web`) receives only metrics (CSV/JSON) and model weights (`.safetensors`).
- The notebook never prints page contents. Don't add cells that print, display or render a page, and never open a URL from the data.
- When you're finished: Runtime → **Disconnect and delete runtime**.

In [ ]:
# Setup. Runs on Colab, or locally inside WSL via notebooks/run_local_wsl.sh (same code, different folders).
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # pin the libraries the training code depends on; keep Colab's own torch/CUDA (the local script installs the same pins)
    !pip install -q "transformers==4.57.6" "accelerate==1.15.0"
import hashlib, io, json, os, shutil, urllib.request, zipfile
from glob import glob
import numpy as np, pandas as pd, sklearn, torch, transformers, accelerate
assert transformers.__version__ == "4.57.6", "an older transformers is still loaded: Runtime > Restart session, then Run all"
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
VERSIONS = dict(python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda, transformers=transformers.__version__,
                accelerate=accelerate.__version__, sklearn=sklearn.__version__, pandas=pd.__version__, gpu=GPU)
pd.set_option("display.width", 200, "display.max_columns", 20)
print(VERSIONS)

In [ ]:
# Settings
SEEDS = [42]   # one training run per seed. Add 43, 44 in later sessions: finished seeds are skipped.
EPOCHS = 10    # upstream: 10. Training stays FP32 like upstream: MobileBERT's logits reach ~1e6, beyond FP16's range.
if IN_COLAB:
    DATA = "/content/phish-data"                  # this VM's disk only: never Google Drive, never your PC
    OUT = "/content/drive/MyDrive/phishlang_web"  # metrics and model weights only
else:  # local run inside WSL Ubuntu: the Linux disk only, never a Windows drive (/mnt/...)
    DATA = os.path.expanduser("~/phish-research/data")
    OUT = os.path.expanduser("~/phish-research/out")
    assert not os.path.realpath(DATA).startswith("/mnt/"), "phishing data must stay on the Linux disk"
UPSTREAM = "https://raw.githubusercontent.com/UTA-SPRLab/phishlang/6b7283854ef8a6edc73f945c1ce7e51b2e1dc4fd/"
BASE = dict(pretrained_model_name_or_path="google/mobilebert-uncased",
            revision="a34c332593aea210dcccd7eaacdb3c9f62aa2700")  # Hugging Face's safetensors conversion of the weights (PR #8)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Download the upstream files (parsed pages + released model) and verify each one against its git blob hash.
FILES = {
    "training_data/parsed_samples_part_aa": "c86a4f13001dea77d52661623ad3b25bc27ec25d",
    "training_data/parsed_samples_part_ab": "d0da9563a4f1ebf753ad08c65d4005f4449ec963",
    "training_data/parsed_samples_part_ac": "f569fc88dc0d68ce19c97f3e93dd9c6db6bbe35d",
    "training_data/parsed_samples_part_ad": "a5d86de3088dc8a52c07acac2c2804c1f0ee7edd",
    "training_data/parsed_samples_part_ae": "73358bdb721598038dfa18e0f02e424b1804af4c",
    "src/model/config.json": "63db0e118287ca50da495aed6a9d09f73bc12b0b",
    "src/model/model.safetensors": "6f0d59084890336401ece42c86cdb0e48f22c2ef",
    "src/model/special_tokens_map.json": "a8b3208c2884c4efb86e49300fdd3dc877220cdf",
    "src/model/tokenizer_config.json": "b68a4fc7fa1b9314f3bf91e6590fa8c53bf5e185",
    "src/model/vocab.txt": "fb140275c155a9c7c5a3b3e0e77a9e839594a938",
}

def git_blob_sha1(path):
    h = hashlib.sha1(b"blob %d\0" % os.path.getsize(path))
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for rel, sha in FILES.items():
    path = f"{DATA}/{rel}"
    if not (os.path.exists(path) and git_blob_sha1(path) == sha):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(UPSTREAM + rel, path)
    assert git_blob_sha1(path) == sha, f"hash mismatch: {rel}"
os.makedirs(f"{OUT}/results", exist_ok=True)
print(f"all {len(FILES)} upstream files verified")

In [ ]:
# Read the pages straight from the archive (never unpacked to disk), 64 at a time. For each page keep only what later
# steps need: a hash of its text, its length in tokens (E0) and its first 128 tokens (the model input, as in upstream
# training). Page text is dropped right away, so RAM stays low even though some pages are very long.
from transformers import AutoTokenizer, MobileBertTokenizer
transformers.logging.set_verbosity_error()  # hides "longer than the maximum length" warnings: we truncate ourselves
tok = AutoTokenizer.from_pretrained(**BASE)

def first_128(ids):
    # same as tok(text, truncation=True, max_length=128): [CLS] + first 126 tokens + [SEP] (checked in the next cell)
    return ids if len(ids) <= 128 else ids[:127] + [tok.sep_token_id]

records, inputs, check_texts, batch, folders, skipped = [], [], [], [], {}, 0
def add_batch():
    ids_batch = tok([text for _, _, text in batch], return_token_type_ids=False, return_attention_mask=False)["input_ids"]
    for (page_id, label, text), ids in zip(batch, ids_batch):
        sha = hashlib.sha1(text.encode()).hexdigest()
        records.append((page_id, label, sha, not text.strip(), len(ids)))
        inputs.append(first_128(ids))
        if sha.startswith("00"):  # about 1 page in 256: first 5,000 characters kept for the tokenizer checks
            check_texts.append(text[:5000])
    batch.clear()

# Rebuild the archive as the upstream README says (cat parsed_samples_part_* > parsed_samples_combined.zip).
combined = f"{DATA}/parsed_samples_combined.zip"
with open(combined, "wb") as out:
    for rel in sorted(r for r in FILES if r.startswith("training_data/")):
        with open(f"{DATA}/{rel}", "rb") as part:
            shutil.copyfileobj(part, out)
with zipfile.ZipFile(combined) as outer:
    for inner_name, label, prefix in [("parsed_phish_samples.zip", 1, "phish/"), ("parsed_benign_samples.zip", 0, "benign/")]:
        with zipfile.ZipFile(io.BytesIO(outer.read(inner_name))) as inner:
            for info in inner.infolist():
                if info.is_dir():
                    continue
                if not info.filename.endswith(".txt") or "__MACOSX" in info.filename:
                    skipped += 1
                    continue
                folder = prefix + os.path.dirname(info.filename)
                folders[folder] = folders.get(folder, 0) + 1
                # decoded like upstream's read_text: UTF-8, undecodable bytes dropped
                batch.append((prefix + info.filename, label, inner.read(info).decode("utf-8", errors="ignore")))
                if len(batch) == 64:
                    add_batch()
add_batch()
pages = pd.DataFrame(records, columns=["id", "label", "text_sha1", "blank", "n_tokens"])
assert pages.id.is_unique
CLASS = {1: "phishing", 0: "benign"}
print("pages per archive folder:", folders, "| other entries skipped:", skipped)
print("pages read:", pages.label.map(CLASS).value_counts().to_dict(), "| paper section 3.2.1: 22,419 phishing, ~26K benign")

# Clean: drop empty pages, pages whose exact text is in both classes, and exact duplicates within a class.
both = pages.groupby("text_sha1").label.transform("nunique").gt(1) & ~pages.blank
kept = pages[~pages.blank & ~both].sort_values("id").drop_duplicates("text_sha1")
print(f"removed: {pages.blank.sum()} empty, {both.sum()} with text in both classes, {(~pages.blank & ~both).sum() - len(kept)} exact duplicates")
input_ids = torch.full((len(kept), 128), tok.pad_token_id)
attention_mask = torch.zeros_like(input_ids)
for row, i in enumerate(kept.index):
    input_ids[row, :len(inputs[i])] = torch.tensor(inputs[i])
    attention_mask[row, :len(inputs[i])] = 1
enc = dict(input_ids=input_ids, token_type_ids=torch.zeros_like(input_ids), attention_mask=attention_mask)
df = kept.drop(columns="blank").reset_index(drop=True)
del inputs
print("pages kept:", df.label.map(CLASS).value_counts().to_dict())

In [ ]:
# E0: length of every kept page in tokens (including [CLS] and [SEP]), with MobileBERT's tokenizer.
# share_under_128: pages for which upstream's sliding-window inference (128-token windows) has no full window.
# share_over_128 / share_over_512: pages that 128-token training / MobileBERT's 512 positions cannot see in full.
e0 = f"{OUT}/results/e0_token_length_summary.csv"
if not os.path.exists(e0):
    g = df.groupby(df.label.map(CLASS)).n_tokens
    pd.DataFrame({"pages": g.size(), "median": g.median(), "p90": g.quantile(0.9),
                  "share_under_128": g.apply(lambda s: (s < 128).mean()),
                  "share_over_128": g.apply(lambda s: (s > 128).mean()),
                  "share_over_512": g.apply(lambda s: (s > 512).mean())}).to_csv(e0)
    df[["id", "label", "n_tokens"]].to_csv(f"{OUT}/results/e0_token_lengths.csv", index=False)
print(pd.read_csv(e0, index_col=0).round(3))

# Checks on the sampled pages: the fast tokenizer matches the slow ones upstream used, and first_128 matches real truncation.
for name, slow in [("training (google/mobilebert-uncased)", MobileBertTokenizer.from_pretrained(**BASE)),
                   ("inference (released model)", MobileBertTokenizer.from_pretrained(f"{DATA}/src/model"))]:
    same = sum(slow(t)["input_ids"] == tok(t)["input_ids"] for t in check_texts)
    print(f"same ids as upstream's {name} tokenizer on {same}/{len(check_texts)} pages (first 5,000 characters)")
same = sum(first_128(tok(t)["input_ids"]) == tok(t, truncation=True, max_length=128)["input_ids"] for t in check_texts)
print(f"first_128 equals the tokenizer's own truncation on {same}/{len(check_texts)} pages")

In [ ]:
labels = torch.tensor(df.label.to_numpy())
df["input_sha1"] = [hashlib.sha1(ids.numpy().tobytes()).hexdigest() for ids in enc["input_ids"]]

# Leakage-free split: 70% train / 30% test (the paper's ratio), then 10% of train held out for per-epoch validation.
from sklearn.model_selection import train_test_split
rest, test = train_test_split(df.index, test_size=0.30, stratify=df.label, random_state=42)
train, val = train_test_split(rest, test_size=0.10, stratify=df.label[rest], random_state=42)
df["split"] = "train"
df.loc[val, "split"] = "val"
df.loc[test, "split"] = "test"
split_csv = f"{OUT}/results/split.csv"
if os.path.exists(split_csv):  # a later session must rebuild exactly the same split
    saved = pd.read_csv(split_csv)
    assert saved.id.tolist() == df.id.tolist() and saved.split.tolist() == df.split.tolist(), "split differs from the saved one"
else:
    df[["id", "label", "split", "text_sha1", "input_sha1"]].to_csv(split_csv, index=False)

df["input_in_train"] = df.split.ne("train") & df.input_sha1.isin(set(df.input_sha1[df.split == "train"]))
test_df = df[df.split == "test"]
print(df.groupby("split").label.agg(pages="size", phishing_share="mean").round(3))
print("test pages whose exact text is also in train:", test_df.text_sha1.isin(set(df.text_sha1[df.split == "train"])).sum())
print("test pages whose 128-token input is identical to a training page's:",
      test_df.groupby(test_df.label.map(CLASS)).input_in_train.agg(["sum", "mean"]).round(3).to_dict("index"))
print("distinct 128-token inputs that occur with both labels:", int((df.groupby("input_sha1").label.nunique() > 1).sum()))

In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from transformers import MobileBertForSequenceClassification

@torch.no_grad()
def predict(model, rows):
    # phishing probability for the given df rows
    model.to(DEVICE).eval()
    probs = []
    for i in range(0, len(rows), 256):
        batch = {k: v[rows[i:i + 256]].to(DEVICE) for k, v in enc.items()}
        probs.append(torch.softmax(model(**batch).logits.float(), -1)[:, 1].cpu())
    return torch.cat(probs).numpy()

def metrics(y, p):
    # precision/recall/F1 of the phishing class; weighted F1 is what upstream's script reports
    pred = (p > 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(pages=len(y), accuracy=accuracy_score(y, pred), precision=precision, recall=recall, f1=f1,
                weighted_f1=precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)[2],
                pr_auc=average_precision_score(y, p), roc_auc=roc_auc_score(y, p), tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))

def evaluate(model, out_dir):
    # scores all test pages, and separately the test pages whose 128-token input never occurs in train
    p = predict(model, test_df.index.to_numpy())
    test_df.assign(prob_phishing=p)[["id", "label", "input_in_train", "prob_phishing"]].to_csv(f"{out_dir}/test_predictions.csv", index=False)
    y, unseen = test_df.label.to_numpy(), ~test_df.input_in_train.to_numpy()
    with open(f"{out_dir}/test_metrics.json", "w") as f:
        json.dump({"all_test_pages": metrics(y, p), "unseen_input_pages": metrics(y[unseen], p[unseen]), "versions": VERSIONS}, f, indent=2)

def show(out_dir):
    with open(f"{out_dir}/test_metrics.json") as f:
        m = json.load(f)
    print(pd.DataFrame({k: v for k, v in m.items() if k != "versions"}).T.round(4))

# E1 sanity check: the released PhishLang model on our test split.
# NOT a held-out score: the released model was probably trained on most of these pages.
e1 = f"{OUT}/results/e1_released_model"
if not os.path.exists(f"{e1}/test_metrics.json"):
    os.makedirs(e1, exist_ok=True)
    evaluate(MobileBertForSequenceClassification.from_pretrained(f"{DATA}/src/model", use_safetensors=True), e1)
show(e1)

In [ ]:
# E2: retrain MobileBERT with upstream's hyper-parameters on the leakage-free split.
from transformers import Trainer, TrainingArguments, set_seed
from transformers.trainer_utils import get_last_checkpoint

class Pages(torch.utils.data.Dataset):
    def __init__(self, split):
        self.rows = np.flatnonzero(df.split.to_numpy() == split)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {**{k: v[r] for k, v in enc.items()}, "labels": labels[r]}

def val_metrics(p):
    return metrics(p.label_ids, torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy())

for seed in SEEDS:
    run = f"{OUT}/runs/e2_seed{seed}"
    if os.path.exists(f"{run}/test_metrics.json"):
        print("already finished:", run)
        continue
    set_seed(seed)  # also makes the new classifier head reproducible
    model = MobileBertForSequenceClassification.from_pretrained(**BASE, use_safetensors=True, num_labels=2)
    args = TrainingArguments(output_dir=run, seed=seed, num_train_epochs=EPOCHS, learning_rate=2e-5, weight_decay=0.01,
                             per_device_train_batch_size=32, gradient_accumulation_steps=4, per_device_eval_batch_size=256,
                             eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                             logging_steps=50, report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=Pages("train"), eval_dataset=Pages("val"), compute_metrics=val_metrics)
    resume = get_last_checkpoint(run) if os.path.isdir(run) else None
    print(f"seed {seed}:", f"resuming from {resume}" if resume else "starting")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(f"{run}/final_model")  # model.safetensors + config.json
    tok.save_pretrained(f"{run}/final_model")
    with open(f"{run}/log_history.json", "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)
    evaluate(trainer.model, run)
    for checkpoint in glob(f"{run}/checkpoint-*"):  # final_model is all we keep
        shutil.rmtree(checkpoint)
    show(run)

In [ ]:
# Summary: E1 (released model, not held out) and every finished E2 seed.
results = {"E1 released model": f"{OUT}/results/e1_released_model/test_metrics.json"}
results.update({f"E2 seed {s}": f"{OUT}/runs/e2_seed{s}/test_metrics.json" for s in SEEDS})
rows = {}
for name, path in results.items():
    if os.path.exists(path):
        with open(path) as f:
            m = json.load(f)
        rows.update({(name, subset): m[subset] for subset in ("all_test_pages", "unseen_input_pages")})
cols = ["pages", "accuracy", "precision", "recall", "f1", "weighted_f1", "pr_auc", "roc_auc"]
table = pd.DataFrame(rows).T[cols]
table.to_csv(f"{OUT}/results/summary.csv")
print(table.round(4))
e2 = table[[name.startswith("E2") for name, _ in table.index]]
if len(e2) > 2:  # at least two seeds: mean and standard deviation per test subset
    print(e2.astype(float).groupby(level=1).agg(["mean", "std"]).round(4))
print("Paper Table 2 (MobileBERT): accuracy 0.96, precision 0.95, recall 0.96, F1 0.96")

In [ ]:
# Colab: delete the phishing data from this VM now (Runtime > Disconnect and delete runtime does the same).
# Locally the data stays in the quarantined ~/phish-research/data for later experiments.
if IN_COLAB:
    shutil.rmtree(DATA, ignore_errors=True)
    print("deleted", DATA)